# Complexity and Compactness

Compare attribution concentration with separate faithfulness evidence. Sparse does not automatically mean correct.

In [ ]:
import numpy as np
import pandas as pd
from xai_eval.data import breast_cancer_split
from xai_eval.models import make_models, positive_probability
from xai_eval.explainers import local_perturbation_attribution
from xai_eval.metrics import entropy_complexity, gini_sparsity, deletion_auc

x_train, x_test, y_train, _, _ = breast_cancer_split(random_state=42)
baseline = np.median(x_train, axis=0)
model = make_models(random_state=42)["LogisticRegression"].fit(x_train, y_train)
x = x_test[0]
score_fn = lambda point: positive_probability(model, point)
concentrated = np.array([0.91, 0.04, 0.03, 0.01, 0.01])
diffuse = np.repeat(0.2, 5)
proxy_examples = pd.DataFrame([
    {"pattern": "concentrated", "entropy": entropy_complexity(concentrated), "gini": gini_sparsity(concentrated)},
    {"pattern": "diffuse", "entropy": entropy_complexity(diffuse), "gini": gini_sparsity(diffuse)},
])
proxy_examples

In [ ]:
from xai_eval.metrics import deletion_curve

fractions, scores = deletion_curve(score_fn, x, attribution, baseline=baseline, steps=20)
assert len(fractions) == len(scores)
assert 0.0 <= fractions[0] < fractions[-1] <= 1.0
assert np.isfinite(scores).all()
print("Deletion curve validated; sparsity did not substitute for this check.")

In [ ]:
attribution = local_perturbation_attribution(model, x, baseline=baseline)
faithfulness_deletion_auc = deletion_auc(score_fn, x, attribution, baseline=baseline)
actual = pd.DataFrame([{
    "pattern": "model explanation",
    "entropy": entropy_complexity(attribution),
    "gini": gini_sparsity(attribution),
    "deletion_auc": faithfulness_deletion_auc,
}])
print("Concentration is descriptive; deletion AUC tests a different property.")
actual